# Car Price Prediction with Machine Learning

This notebook documents an end-to-end machine learning workflow for car price prediction: data cleaning, EDA, feature engineering, model comparison, evaluation, and prediction.

## 1. Import Libraries and Project Modules

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT_DIR / 'src'))

from data_preprocessing import clean_dataset, load_dataset, split_features_target, add_engineered_features
from train_model import train_and_save, DATA_PATH

sns.set_theme(style='whitegrid', palette='viridis')

## 2. Load Dataset

In [ ]:
df_raw = load_dataset(DATA_PATH)
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include='all')

## 3. Data Cleaning

The preprocessing module removes duplicates, standardizes text columns, validates numeric fields, handles invalid records, and prepares missing values for the Scikit-learn pipeline.

In [ ]:
df = clean_dataset(df_raw)
print(f'Raw rows: {len(df_raw):,}')
print(f'Clean rows: {len(df):,}')
print(f'Duplicates after cleaning: {df.duplicated().sum()}')
df.isna().sum()

## 4. Feature Engineering

In [ ]:
df_featured = add_engineered_features(df)
df_featured[['year', 'mileage', 'engine_size', 'horsepower', 'car_age', 'power_to_engine_ratio', 'mileage_per_year', 'price']].head()

## 5. Advanced Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(11, 7))
sns.heatmap(df_featured.select_dtypes(include='number').corr(), annot=True, cmap='viridis', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(df['price'], kde=True, bins=35, ax=axes[0], color='#2563eb')
axes[0].set_title('Price Distribution')
sns.scatterplot(data=df, x='mileage', y='price', hue='fuel_type', alpha=0.7, ax=axes[1])
axes[1].set_title('Mileage vs Price by Fuel Type')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
brand_price = df.groupby('brand')['price'].median().sort_values(ascending=False)
sns.barplot(x=brand_price.index, y=brand_price.values, hue=brand_price.index, palette='mako', legend=False)
plt.title('Median Price by Brand')
plt.xticks(rotation=35, ha='right')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='transmission', y='price', hue='fuel_type')
plt.title('Price Spread by Transmission and Fuel Type')
plt.show()

## 6. Model Training and Comparison

The training script evaluates Linear Regression, Decision Tree Regressor, Random Forest Regressor, and Gradient Boosting Regressor using a holdout test set.

In [ ]:
best_pipeline, results = train_and_save()
results

## 7. Model Selection

The model with the highest R2 Score is selected as the final model. MAE, MSE, and RMSE are also reported to understand absolute prediction error.

In [ ]:
best_model = results.iloc[0]
print(f"Best model: {best_model['Model']}")
print(f"R2 Score: {best_model['R2 Score']:.4f}")
print(f"MAE: {best_model['MAE']:,.2f}")
print(f"RMSE: {best_model['RMSE']:,.2f}")

## 8. Prediction Example

In [ ]:
from predict import predict_price

predicted_price = predict_price(
    brand='Toyota',
    year=2020,
    mileage=42000,
    fuel_type='Petrol',
    transmission='Automatic',
    engine_size=2.0,
    horsepower=160,
)
print(f'Predicted price: ${predicted_price:,.2f}')

## 9. Generated Project Artifacts

After running the workflow, the repository contains a trained model, evaluation report, model metrics CSV, EDA charts, model comparison chart, and feature impact visualization.